# Urban Sentinel
## From public records to an inspection priority
A **2–3 minute presentation**, computed in this notebook. Municipal records power the model; satellite movement adds measured context. This is decision support for façade inspections, **not a collapse forecast**.

**Before recording:** select your LinuxONE ML kernel, restart it, and Run All. Download, evaluation and training can take many minutes. Start recording at **01** after every cell succeeds. Keep executed copies under the ignored run folder; never commit data or credentials.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, sys

REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ml/urban_sentinel").is_dir()), None)
if REPO is None:
    raise RuntimeError("Open this notebook inside the megatron checkout; see ml/notebooks/README.md.")
RUN_ID = os.environ.get("URBAN_SENTINEL_PRESENTATION_RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
RUN_DIR = REPO / "data/presentation" / RUN_ID
REBUILD_SATELLITE = False  # Expensive, authenticated; separate environment required.
SATELLITE_PYTHON = os.environ.get("SATELLITE_PYTHON")
sys.path.insert(0, str(REPO / "ml/notebooks"))
import presentation as p
manifest = p.configure(REPO, RUN_DIR)
import charts as viz
viz.theme()

In [ ]:
# Rebuild by default. A completed, hash-checked source snapshot can be reused after a kernel restart.
prepared = p.prepare_data()
cutoff_check = p.demonstrate_cutoff(prepared["context"])
result = p.rebuild()

# 01 · Running here, not just claimed
This is the **current kernel's** platform and newly trained model. `s390x` identifies IBM Z architecture; it does not alone establish a specific LinuxONE instance. Show your instance's Jupyter/server page alongside this card. Saved LinuxONE evidence is labeled separately in the appendix.

In [ ]:
viz.current_platform(result["manifest"])
viz.card("Model trained in this run", result["manifest"]["model"]["model_sha256"],
         f"Completed {result['manifest']['completed_at']} · {result['manifest']['n_scored']:,} pending buildings scored")

# 02 · Public records → a usable building history
Façade filings provide the target and prior history. Dated DOB/ECB violations and PLUTO building attributes add context. Complaints were evaluated but are **not deployed model inputs**. Footprints and satellite observations support the map, not the deployed predictor.

In [ ]:
viz.preparation(prepared)

In [ ]:
viz.table(prepared["raw"][["bin", "cycle", "submitted_on", "filing_status"]].head(3))
viz.table(prepared["context"].fac[["bin", "cycle", "observed_on", "severity"]].head(3))

# 03 · A fair historical question
Can pre-cutoff records prioritize buildings later rated **UNSAFE among Initial FISP filings**? The production label takes the worst classified Initial rating within a building/cycle; amended and subsequent filings are not target labels. Unfiled outcomes are unknown—not SAFE.

Features exclude target-cycle filings and future dated events. The checks remove future records and alter future outcomes; history/event features must not change. PLUTO and building-to-lot links remain current snapshots, not a complete historical reconstruction.

In [ ]:
viz.timeline(cutoff_check)

# 04 · Results an inspector can use
Every method gets the **same building cohort and 500-inspection budget**. The DOB benchmark is a 2022 published snapshot, not a clean pre-2020 forecast. These are retrospective recorded outcomes, not verified future incidents.

In [ ]:
viz.precision(result["metrics"])

In [ ]:
viz.reliability(result["metrics"])

# 05 · What did satellite movement add?
NASA OPERA provides measured line-of-sight ground displacement—not direct façade damage. The saved tax-block-grouped ablation found no demonstrated improvement from adding movement. **Satellite stays a visualization layer, not a deployed predictor.** Optional reconstruction is disabled by default; the plot identifies rebuilt versus saved measurements.

In [ ]:
import satellite
satellite_run = satellite.run(REPO, RUN_DIR, SATELLITE_PYTHON, enabled=REBUILD_SATELLITE)
satellite_metadata = viz.satellite(REPO)

# 06 · A prediction computed in this kernel
Choose the highest-ranked pending building previously recorded SAFE; otherwise use the highest-ranked pending building. Load this run's model, rebuild the features, and attempt **parity-checked NNPA inference**. If hardware or compatible software is unavailable, explicitly show XGBoost CPU execution.

In [ ]:
live, factors = p.live_prediction(result, prepared["context"])
viz.prediction(live, factors)

## Turn records into a better inspection queue
A transparent historical test, an explainable current score, and measured geographic context—ready for **qualified field inspection**. Probability means a recorded UNSAFE façade rating, not the probability of structural collapse.

**End of the short recording.** The technical appendix follows.

# Appendix A · Data inventory and parsing
Municipal counts below are from this run. Event counts are retained façade-relevant records **after source filters**, not complete citywide source sizes. PLUTO is queried for building-linked lots. Footprints are a committed Lower Manhattan export; satellite metadata describes a separate observation window.

In [ ]:
inventory = p.source_inventory(REPO, prepared)
viz.table(inventory, limit=10)
viz.table(prepared["audit"], limit=10)
viz.card("Municipal snapshot", prepared["receipt"]["completed_at"],
         "Raw files are SHA-256 checked before and after evaluation / scoring. This is one acquisition window, not an atomic citywide snapshot.")

### Parsing choices
BINs remain seven-digit strings; invalid and placeholder BINs are removed. Cycles become integers; unparseable dates become missing. A filing is observable on the later of submission/filing dates; undated auto-generated No Report Filed rows use the subcycle deadline. SAFE/SWARMP/UNSAFE map to 0/1/2; unknown statuses do not become SAFE.

DOB and ECB events use dated façade-relevant code filters. Complaints are parsed but excluded from deployed features. Building attributes join via a derived ten-digit BBL, sometimes a condominium billing lot. Missing fields are retained for production missing-value handling.

In [ ]:
import pandas as pd
for source, events in prepared["context"].events.items():
    viz.card(source.upper(), f"{len(events):,} retained records",
             f"Date range: {events.date.min()} to {events.date.max()} · missing dates: {events.date.isna().sum():,}")
    viz.table(events.head(3))
viz.table(prepared["pluto"].reset_index()[["bbl", "yearbuilt", "numfloors", "latitude", "longitude"]].head(4))

In [ ]:
footprints = json.loads((REPO / "frontend/public/data/buildings-demo.geojson").read_text())
viz.table(pd.DataFrame([{"Geometry": f["geometry"]["type"], **f["properties"]}
                        for f in footprints["features"][:3]]).iloc[:, :6])
viz.table(pd.DataFrame([{"Metadata": k, "Value": str(satellite_metadata.get(k))}
                       for k in ["observation_start", "observation_end", "n_epochs", "pixel_size_m", "frame", "quality_filters"]]))

# Appendix B · Metric audit and limitations
Primary evaluation trains on cycles 7–8 and tests cycle 9 at **2020-02-21**. The deployed pending-building model is separately trained at the current cycle's elapsed time; its probability is not the historical evaluation model's score. Calibration uses the production temporal design.

Previously-SAFE results use their own common subgroup and top-250 budget; do not compare that percentage directly with full-cohort top-500 results. The last-status rule can have ties. DOB's 2022 information timing can favor that benchmark.

Selection bias remains: FISP covers eligible buildings with filings; absent labels are not negatives. Filing status is a regulatory outcome, not collapse. Refreshed historical records can contain corrections. PLUTO and BIN-to-BBL mappings are not reconstructed historically. Satellite pixels are shared by buildings, line-of-sight movement is not vertical motion without assumptions, and 2016–2020 observations are not current movement.

In [ ]:
viz.table(viz.comparison_rows(result["metrics"]))
viz.table(pd.DataFrame(result["metrics"]["primary_cycle9"]["calibration"]["reliability"]), limit=15)
viz.table(pd.DataFrame([{"Check": k, "Result": str(v)} for k, v in cutoff_check.items()]))
viz.table(factors)

# Appendix C · Saved LinuxONE evidence (not this execution)
Earlier LinuxONE measurements below never substitute for a failed current-run training/inference. Show your actual instance separately. Earlier exploratory NNPA benchmarks include incompatible missing-value variants; the current inference cell uses only the production parity gate.

In [ ]:
saved = json.loads((REPO / "ml/reports/linuxone/latency_s390x.json").read_text())
h = saved["health"]
viz.card("Historical LinuxONE evidence · NOT this kernel",
         f"{h['platform']['machine']} · {h['platform']['system']} · model {h['model']}",
         f"Captured {h['platform']['timestamp_utc']} · scores as of {h['data_as_of']} · {saved['buildings']} timed buildings")
viz.table(pd.DataFrame([{"Engine": engine, "Median model ms": values["model_inference_ms"]["median"],
                       "Median end-to-end ms": values["end_to_end_ms_incl_feature_build"]["median"]}
                      for engine, values in saved["engines"].items()]))
viz.card("Current run remains separate", live["platform"]["machine"],
         f"{live['platform']['timestamp_utc']} · engine {live['engine']} · model {live['model']}")

# Appendix D · Optional satellite reconstruction
Set `REBUILD_SATELLITE = True` in setup and provide `SATELLITE_PYTHON` as an absolute path to a **separate** environment, then restart and Run All. Section 05 executes the stages below and updates its plot and ablation.

1. Account owner configures Earthdata `~/.netrc` privately, fills in Study Area and authorizes ASF access. Never put a password, token or signed URL in a cell.
2. Preflight imports compiled GDAL/rasterio, HDF5/h5py, PROJ and other dependencies in the separate environment and checks free disk.
3. Fetch FISP building locations; download frame 8622 subsets over NYC; fetch perpendicular baselines.
4. Extract pre-2020-02-21 movement features and pixel velocities; generate quality figures/report and measured map export.
5. Run the existing ablation in the **ML environment**, using this run's municipal snapshot.

Allow hours, roughly 2 GB of subsets, at least 8 GiB free disk and several GiB RAM. Child logs are suppressed because download exceptions may contain signed URLs; only stage names, timings and safe platform metadata are saved. See `ml/insar/README.md` for troubleshooting.

**Compatibility boundary:** the geospatial workflow was developed on a laptop. LinuxONE compiled dependencies and authenticated execution require verification on your chosen environment. The default municipal run does not require Earthdata credentials.

In [ ]:
viz.card("Satellite reconstruction status", satellite_run["status"], satellite_run.get("note", "See satellite_manifest.json."))
viz.table(pd.DataFrame([{"Stage": label, "Interpreter": "ML kernel" if command[1] == "-m" else "Separate geospatial Python"}
                       for label, command in satellite.commands(REPO, RUN_DIR, SATELLITE_PYTHON or "/configure/separate/python")]), limit=10)

# Appendix E · Recording script and reproduction
| Time | Stop | Narration |
|---|---|---|
| 0:00–0:20 | 01 | “Here is our kernel's architecture, execution timestamp and the model we just trained.” |
| 0:20–0:45 | 02 | “Filings, violations and attributes become a building history. Missing fields remain visible.” |
| 0:45–1:05 | 03 | “We ask what could have been known before inspection, and test on a later cycle.” |
| 1:05–1:40 | 04 | “For the same 500 inspections, these are the recorded UNSAFE outcomes. We check previously-SAFE buildings and calibration too.” |
| 1:40–2:10 | 05 | “Movement is measured, but did not demonstrate added predictive value. It stays contextual.” |
| 2:10–2:40 | 06 | “This score was computed here. We show the actual engine, measured latency and contributing features.” |
| 2:40–2:50 | Closing | “A priority for qualified field inspection—not a collapse prediction.” |

Show the actual LinuxONE instance/Jupyter UI first. Record after successful execution with inputs collapsed, or export HTML with `--no-input`. Scrolling saved outputs is not live recomputation. Adjust narration to the metrics and engine actually shown.

From the repository root, after setup in `ml/notebooks/README.md`:

```bash
export URBAN_SENTINEL_PRESENTATION_RUN_ID=linuxone-recording
mkdir -p data/presentation/linuxone-recording
python -m nbconvert --execute --to notebook --ExecutePreprocessor.kernel_name=urban-sentinel --ExecutePreprocessor.timeout=7200 \
  --output Urban_Sentinel_LinuxONE.executed --output-dir data/presentation/linuxone-recording \
  ml/notebooks/Urban_Sentinel_LinuxONE.ipynb
python -m nbconvert --to html --no-input \
  --output Urban_Sentinel_LinuxONE --output-dir data/presentation/linuxone-recording \
  data/presentation/linuxone-recording/Urban_Sentinel_LinuxONE.executed.ipynb
```

A new run ID refreshes sources; an existing ID reuses its hash-checked snapshot. Keep executed notebooks, HTML, figures, datasets, models, reports and manifests inside ignored `data/presentation/`. The source notebook is committed without outputs. Local verification **does not establish LinuxONE execution**: run there and retain that platform evidence.